In [15]:
import trafpy.generator as tpg
from trafpy.benchmarker import BenchmarkImporter

In [16]:
num_eps = 4

In [17]:
racks_dict, num_racks = {}, 2
eps_per_rack = int(num_eps/num_racks)
for rack in range(num_racks):
    racks_dict[rack] = [ep for ep in range(rack*eps_per_rack, (rack*eps_per_rack)+eps_per_rack)]
print(racks_dict)

{0: [0, 1], 1: [2, 3]}


In [18]:
ep_capacity = 400000

In [19]:
net = tpg.gen_arbitrary_network(num_eps=num_eps, ep_capacity=ep_capacity, racks_dict=racks_dict)

In [20]:
print(net.graph.keys())
print(net.graph['topology_type'])

dict_keys(['endpoints', 'endpoint_label', 'num_channels_per_link', 'ep_link_capacity', 'ep_link_port_capacity', 'max_nw_capacity', 'curr_nw_capacity_used', 'num_active_connections', 'total_connections_blocked', 'node_labels', 'topology_type', 'channel_names', 'rack_to_ep_dict', 'ep_to_rack_dict'])
arbitrary_endpoints_4_chancap_400000_channels_1


In [21]:
print("max_nw_capacity:", net.graph['max_nw_capacity'])
print("topology_type:", net.graph['topology_type'])
print("endpoints:", net.graph['endpoints'])
print("endpoint_label:", net.graph['endpoint_label'])
print("num_channels_per_link:", net.graph['num_channels_per_link'])
print("ep_link_capacity:", net.graph['ep_link_capacity'])
print("ep_link_port_capacity:", net.graph['ep_link_port_capacity'])
print("curr_nw_capacity_used:", net.graph['curr_nw_capacity_used'])
print("num_active_connections:", net.graph['num_active_connections'])
print("total_connections_blocked:", net.graph['total_connections_blocked'])
print("node_labels:", net.graph['node_labels'])
print("channel_names:", net.graph['channel_names'])
print("rack_to_ep_dict:", net.graph['rack_to_ep_dict'])
print("ep_to_rack_dict:", net.graph['ep_to_rack_dict'])


max_nw_capacity: 800000.0
topology_type: arbitrary_endpoints_4_chancap_400000_channels_1
endpoints: ['0', '1', '2', '3']
endpoint_label: None
num_channels_per_link: 1
ep_link_capacity: 400000
ep_link_port_capacity: 200000.0
curr_nw_capacity_used: 0
num_active_connections: 0
total_connections_blocked: 0
node_labels: [None]
channel_names: ['channel_1']
rack_to_ep_dict: {'0': ['0', '1'], '1': ['2', '3']}
ep_to_rack_dict: {'0': '0', '1': '0', '2': '1', '3': '1'}


In [22]:
network_load_config = {'network_rate_capacity': net.graph['max_nw_capacity'], 
                       'ep_link_capacity': net.graph['ep_link_capacity'],
                       'target_load_fraction': 0.4}

In [23]:
importer = BenchmarkImporter(benchmark_version='v001', load_prev_dists=False)

load_prev_dist=False. Will re-generate dists with given network params and override any previously saved distributions.


In [24]:
dcn_dist = importer.get_benchmark_dists(benchmark_name='commercial_cloud',eps=net.graph['endpoints'],racks_dict=net.graph['rack_to_ep_dict'])

Set to save benchmark commercial_cloud distribution data to /home/hsd/workspace/trafpy/trafpy/benchmarker/versions/benchmark_v001/benchmarks/commercial_cloud/
Saved node_dist distribution data to /home/hsd/workspace/trafpy/trafpy/benchmarker/versions/benchmark_v001/benchmarks/commercial_cloud/
Saved flow_size_dist distribution data to /home/hsd/workspace/trafpy/trafpy/benchmarker/versions/benchmark_v001/benchmarks/commercial_cloud/
Saved interarrival_time_dist distribution data to /home/hsd/workspace/trafpy/trafpy/benchmarker/versions/benchmark_v001/benchmarks/commercial_cloud/


In [25]:
print(dcn_dist.keys())

dict_keys(['node_dist', 'flow_size_dist', 'interarrival_time_dist', 'num_ops_dist'])


In [26]:
jsd_threshold = 0.9

In [27]:
from pathlib import Path
import gzip
import pickle
import time

path_to_data = 'data/duration_check_test/'
Path(path_to_data).mkdir(exist_ok=True, parents=True)

print('Generating \'{}\' traffic demands for {} network'.format(dcn_dist, net.graph['topology_type']))
    
# get node, flow size, and flow inter-arrival time benchmark dists
dists = dcn_dist
    
# generate traffic demands
demand_data = tpg.create_demand_data(eps=net.graph['endpoints'],
                                     node_dist=dists['node_dist'],
                                     flow_size_dist=dists['flow_size_dist'],
                                     interarrival_time_dist=dists['interarrival_time_dist'],
                                     network_load_config=network_load_config,
                                     jensen_shannon_distance_threshold=jsd_threshold)
                                   #  min_last_demand_arrival_time=2)
    
    # save demands as pickle file
    # filename = path_to_data+'{}_demand_data.pickle'.format(dcn)
    # with gzip.open(filename, 'wb') as f:
    #     pickle.dump(demand_data, f)

Generating '{'node_dist': array([[0.        , 0.20570833, 0.03464583, 0.03464583],
       [0.20570833, 0.        , 0.01722917, 0.01347917],
       [0.03464583, 0.01722917, 0.        , 0.19429167],
       [0.03464583, 0.01347917, 0.19429167, 0.        ]]), 'flow_size_dist': {4625.0: 0.00082, 9775.0: 0.0003, 25.0: 0.05242666666666667, 15575.0: 0.00016666666666666666, 5850.0: 0.00052, 1000.0: 0.004026666666666666, 2800.0: 0.00138, 675.0: 0.005753333333333333, 325.0: 0.01106, 1375.0: 0.0028266666666666666, 50.0: 0.0371, 125.0: 0.02212, 200.0: 0.016126666666666668, 175.0: 0.017406666666666668, 475.0: 0.008026666666666666, 5375.0: 0.0006266666666666666, 294925.0: 6.666666666666667e-06, 100.0: 0.02530666666666667, 1325.0: 0.00304, 725.0: 0.005713333333333333, 85725.0: 2e-05, 7100.0: 0.0004733333333333333, 600.0: 0.00666, 1775.0: 0.0023266666666666666, 5975.0: 0.0005266666666666667, 75.0: 0.030573333333333334, 17675.0: 0.00016666666666666666, 825.0: 0.0048, 15050.0: 0.00012, 500.0: 0.007833333

Packed 6000 flows in 0.288 s | Node distribution Jensen Shannon distance from target achieved: 1.9597655148487675e-08


In [28]:
tpg.save_data_as_csv(data=demand_data,path_to_save=path_to_data+'com_40_nt.csv',overwrite=True)

Time to save data to data/duration_check_test/com_40_nt.csv: 0.01722860336303711 s
